In [1]:
pip install torch torchvision opencv-python scikit-learn tqdm

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
    --------------------------------------- 1.0/44.0 MB 4.6 MB/s eta 0:00:10
   - -------------------------------------- 2.1/44.0 MB 4.2 MB/s eta 0:00:10
   -- ------------------------------------- 3.1/44.0 MB 4.4 MB/s eta 0:00:10
   --- ------------------------------------ 3.7/44.0 MB 4.5 MB/s eta 0:00:09
   ---- ----------------------------------- 4.7/44.0 MB 4.2 MB/s eta 0:00:10
   ----- ---------------------------------- 5.8/44.0 MB 4.3 MB/s eta 0:00:09
   ------ --------------------------------- 6.8/44.0 MB 4.4 MB/s eta 0:00:09
   ------- -------------------------------- 7.9/44.0 MB 4.5 MB/s eta 0:00:08
   ------- -------------------------------- 8.7/44.0 MB 4.5 MB/s eta 0:00:08
   -------- ------------------------------- 8.9/44.0 MB 4.5 MB/s eta 0:00:08
   -------- 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.6 which is incompatible.
pandas 2.2.1 requires numpy<2,>=1.26.0; python_version >= "3.12", but you have numpy 2.4.6 which is incompatible.
streamlit 1.44.1 requires packaging<25,>=20, but you have packaging 26.3 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# -------------------------------------------------------------
# 1. Dataset Class (Extracts sequence of frames from MP4 files)
# -------------------------------------------------------------
class RealLifeViolenceDataset(Dataset):
    def __init__(self, video_paths, labels, sequence_length=16, frame_size=(128, 128)):
        self.video_paths = video_paths
        self.labels = labels
        self.sequence_length = sequence_length
        self.frame_size = frame_size

        # Standard ImageNet normalization for pre-trained CNNs
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(self.frame_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Calculate frame sampling step size to get uniform frame sequences
        step = max(1, total_frames // self.sequence_length)
        frames = []

        for i in range(self.sequence_length):
            frame_idx = i * step
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()

            if not ret or frame is None:
                # Fallback zero frame if reading fails
                frame_tensor = torch.zeros((3, *self.frame_size))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame_tensor = self.transform(frame)

            frames.append(frame_tensor)

        cap.release()

        # Stack into shape: (sequence_length, channels, height, width)
        frames_tensor = torch.stack(frames)
        return frames_tensor, torch.tensor(label, dtype=torch.float32)

# -------------------------------------------------------------
# 2. CNN + LSTM Model Definition
# -------------------------------------------------------------
class CNNLSTMViolenceModel(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=1):
        super(CNNLSTMViolenceModel, self).__init__()
        
        # Use pre-trained MobileNetV2 feature extractor
        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.feature_extractor = mobilenet.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # Freeze CNN backbone to speed up training
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        # MobileNetV2 output dimension is 1280
        feature_dim = 1280

        # Sequence layer: LSTM
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        # Binary classification head
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, C, H, W)
        batch_size, seq_len, C, H, W = x.shape

        # Flatten batch and sequence dimensions for CNN forward pass
        x = x.view(batch_size * seq_len, C, H, W)
        
        features = self.feature_extractor(x)
        features = self.pool(features)
        features = torch.flatten(features, 1) # Shape: (batch_size * seq_len, 1280)

        # Reshape back to sequence format for LSTM
        features = features.view(batch_size, seq_len, -1) # Shape: (batch_size, seq_len, 1280)

        # LSTM forward pass
        lstm_out, (hn, _) = self.lstm(features)
        
        # Use last hidden state output
        out = self.fc(hn[-1])
        return out.squeeze(-1)

# -------------------------------------------------------------
# 3. Training Loop
# -------------------------------------------------------------
def train_model(data_dir, epochs=10, batch_size=8, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using compute device: {device}")

    # Gather video file paths and labels
    video_paths = []
    labels = []

    for label_name, label_val in [("NonViolence", 0), ("Violence", 1)]:
        folder_path = os.path.join(data_dir, label_name)
        if not os.path.exists(folder_path):
            continue
        for file_name in os.listdir(folder_path):
            if file_name.endswith(('.mp4', '.avi')):
                video_paths.append(os.path.join(folder_path, file_name))
                labels.append(label_val)

    print(f"Found total videos: {len(video_paths)}")

    # Split dataset (80% Train, 20% Validation)
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        video_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )

    train_dataset = RealLifeViolenceDataset(train_paths, train_labels)
    val_dataset = RealLifeViolenceDataset(val_paths, val_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Instantiate Model, Loss, and Optimizer
    model = CNNLSTMViolenceModel().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0

        for videos, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            videos, targets = videos.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(videos)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * videos.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct_train += (preds == targets).sum().item()

        epoch_loss = running_loss / len(train_paths)
        epoch_acc = correct_train / len(train_paths)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct_val = 0

        with torch.no_grad():
            for videos, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                videos, targets = videos.to(device), targets.to(device)
                outputs = model(videos)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * videos.size(0)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                correct_val += (preds == targets).sum().item()

        val_loss = val_loss / len(val_paths)
        val_acc = correct_val / len(val_paths)

        print(f"Epoch {epoch+1:02d} | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        # Save model checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_violence_model.pth")
            print(f"-> Saved new best model checkpoint (Val Acc: {val_acc:.4f})")

if __name__ == "__main__":
    # Update this path to where your extracted Kaggle dataset resides
    #DATASET_PATH = "/path/to/real-life-violence-dataset"
    DATASET_PATH = ""  # Update this to your actual folder name/path
    
    train_model(DATASET_PATH, epochs=10, batch_size=8)

Using compute device: cpu
Found total videos: 0


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.